# RAG Query Reformulation — Results Analysis
Visualizes benchmark results from `data/results.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/results.csv')
df.head()

In [ ]:
metrics = ['recall_at_k', 'mrr', 'answer_overlap', 'faithfulness']
summary = df.groupby('pipeline')[metrics].mean().round(3)
print(summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RAG Pipeline Comparison: Baseline vs Query Reformulation Strategies', fontsize=14)

for ax, metric in zip(axes.flatten(), metrics):
    data = df.groupby('pipeline')[metric].mean().sort_values()
    colors = ['#d62728' if 'baseline' in p else '#1f77b4' for p in data.index]
    data.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Score')
    ax.axvline(data['baseline'], color='red', linestyle='--', alpha=0.5, label='baseline')

plt.tight_layout()
plt.savefig('../data/pipeline_comparison.png', dpi=150)
plt.show()

In [ ]:
# Vague vs Specific query breakdown
cat_summary = df.groupby(['pipeline', 'category'])[metrics].mean().round(3)
print('\nVague queries — where reformulation helps most:')
print(cat_summary.xs('vague', level='category')['recall_at_k'].sort_values(ascending=False))

print('\nSpecific queries — reformulation may hurt:')
print(cat_summary.xs('specific', level='category')['recall_at_k'].sort_values(ascending=False))

In [ ]:
# Heatmap: recall_at_k per query per pipeline
pivot = df.pivot_table(index='query_id', columns='pipeline', values='recall_at_k')

plt.figure(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('Recall@k per Query per Pipeline')
plt.tight_layout()
plt.savefig('../data/recall_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Show queries where reformulation helped vs hurt
baseline_recall = df[df['pipeline'] == 'baseline'].set_index('query_id')['recall_at_k']

for strategy in ['enhanced_expansion', 'enhanced_hyde', 'enhanced_stepback', 'enhanced_decomposition']:
    strat_recall = df[df['pipeline'] == strategy].set_index('query_id')['recall_at_k']
    delta = (strat_recall - baseline_recall).dropna()
    print(f'\n{strategy}:')
    print(f'  Improved: {(delta > 0).sum()} queries')
    print(f'  No change: {(delta == 0).sum()} queries')
    print(f'  Hurt: {(delta < 0).sum()} queries')
    print(f'  Mean delta: {delta.mean():.3f}')